# Part 2 · Waste Classification with CNN\n**Owner: Dennis**  ·  Part 2 of the EcoSort Waste Management System\n\nTransfer learning (MobileNetV2), class-imbalance handling, confusion-matrix analysis.

### Environment setup — run first (works in VS Code **and** Google Colab)
Detects where you're running, gets the dataset onto **local disk** (fast — no Google Drive), and unzips it.
In Colab it also clones the repo and installs requirements. Set `DATASET_URL` to the `realwaste.zip` GitHub **Release** link once the repo exists.

In [ ]:
import os, sys, zipfile, pathlib, shutil, subprocess

IN_COLAB = "google.colab" in sys.modules
DATASET_URL = ""  # <-- paste the GitHub Release asset URL for realwaste.zip

if IN_COLAB:
    if not os.path.exists("Ecosort-Waste-Management-System"):
        subprocess.run(["git","clone","https://github.com/Ted-star7/Ecosort-Waste-Management-System.git"], check=False)
    if os.path.isdir("Ecosort-Waste-Management-System"):
        os.chdir("Ecosort-Waste-Management-System")
    subprocess.run(["pip","-q","install","-r","requirements.txt"], check=False)
    if DATASET_URL and not os.path.exists("data/realwaste.zip"):
        subprocess.run(["wget","-q","-O","data/realwaste.zip",DATASET_URL], check=False)

# Unzip RealWaste -> data/RealWaste/<category>/  (idempotent)
if not pathlib.Path("data/RealWaste").exists() and pathlib.Path("data/realwaste.zip").exists():
    with zipfile.ZipFile("data/realwaste.zip") as z:
        z.extractall("data/_tmp")
    for p in pathlib.Path("data/_tmp").rglob("RealWaste"):
        shutil.move(str(p), "data/RealWaste"); break
    shutil.rmtree("data/_tmp", ignore_errors=True)

DATA_DIR = "data"
IMAGE_DIR = os.path.join(DATA_DIR, "RealWaste")
print("Setup OK. Images present:", os.path.isdir(IMAGE_DIR))


In [ ]:
import os, re, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(42); random.seed(42)

# The 9 categories are read from the folder names (sorted) so every part
# uses ONE identical, consistent label order:
IMG_HEIGHT = IMG_WIDTH = 224
BATCH_SIZE = 32

Run **Part 1's pipeline cell** first (or the master notebook top-to-bottom) so `train_ds`, `validation_ds`, `test_dataset`, `class_names` exist.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
num_classes = len(class_names)

data_aug = models.Sequential([layers.RandomFlip("horizontal"),
                              layers.RandomRotation(0.1), layers.RandomZoom(0.1)])
base = tf.keras.applications.MobileNetV2(input_shape=(IMG_HEIGHT,IMG_WIDTH,3),
                                         include_top=False, weights="imagenet")
base.trainable = False
inp = tf.keras.Input((IMG_HEIGHT,IMG_WIDTH,3))
x = data_aug(inp)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(num_classes, activation="softmax")(x)
model = models.Model(inp, out)
model.compile(tf.keras.optimizers.Adam(1e-3), "categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# class weights for imbalance (Plastic ~921 vs Textile Trash ~318)
from sklearn.utils.class_weight import compute_class_weight
y = np.concatenate([y.numpy().argmax(1) for _,y in train_ds])
cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(num_classes), y=y)))
cw

In [ ]:
cbs=[tf.keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True),
     tf.keras.callbacks.ReduceLROnPlateau(patience=2,factor=0.3)]
h1 = model.fit(train_ds, validation_data=validation_ds, epochs=12, class_weight=cw, callbacks=cbs)
# fine-tune top of base
base.trainable=True
for l in base.layers[:-30]: l.trainable=False
model.compile(tf.keras.optimizers.Adam(1e-5),"categorical_crossentropy",metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=validation_ds, epochs=5, class_weight=cw, callbacks=cbs)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
y_true,y_pred=[],[]
for xb,yb in test_dataset:
    p=model.predict(xb,verbose=0); y_pred+=list(p.argmax(1)); y_true+=list(yb.numpy().argmax(1))
print("Test accuracy:", accuracy_score(y_true,y_pred))
print(classification_report(y_true,y_pred,target_names=class_names))
sns.heatmap(confusion_matrix(y_true,y_pred),annot=True,fmt="d",xticklabels=class_names,yticklabels=class_names)
plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()

In [ ]:
def classify_waste_image(image_path):
    """Return predicted waste category for an image path (used by Part 5)."""
    img = tf.keras.utils.load_img(image_path, target_size=(IMG_HEIGHT,IMG_WIDTH))
    arr = tf.keras.utils.img_to_array(img)[None,...]
    return class_names[int(model.predict(arr,verbose=0).argmax(1)[0])]
# model.save("models/cnn_waste_classifier.keras")  # share via GitHub Release

### TODO (Dennis)\n- Justify MobileNetV2 vs EfficientNetB0 (try both)\n- Tune dropout / dense units / fine-tune depth / LR\n- Analyze which categories confuse each other